# 06-指令集、汇编与手动执行程序

> 让已经搭好的 CPU 真正“执行事情”：认识机器指令和汇编语言，把 CPU 接上内存，手动执行 `LOAD_A`，再跟踪一个 `3+5` 程序和显存输出。

## 1. 本篇学习范围

本篇对应课程视频 **61—68**：

- 61_汇编语言和指令集
- 62_内存模块的使用
- 63_CPU与内存的连接
- 64_手动版本CPU的搭建完成
- 65_手动执行第一条机器指令load_A
- 66_手动完成3加5的流程
- 67_c代码执行的解释
- 68_显存工作原理

学习目标是理解：指令如何被编码、CPU 如何从内存取指和读写数据、控制信号如何推动一条指令完成，以及高级语言代码如何逐层变成硬件动作。第 69 个视频开始进入控制器和自动执行，留给第 07 篇。

## 2. 指令、机器码和指令集

CPU 不直接理解“把 3 加 5”这样的自然语言。它只能识别由二进制位组成的机器指令。

- **指令（Instruction）：**告诉 CPU 要做什么以及操作哪些数据；
- **机器码（Machine Code）：**指令在硬件中使用的二进制编码；
- **指令集（Instruction Set）：**CPU 支持的全部指令及其编码、操作数和行为。

一条指令通常包含两部分：

```text
操作码（Opcode） + 操作数/地址字段（Operand）
```

例如，`LOAD_A address` 可以理解为“从某个内存地址取数据，放入 A 寄存器”。课程中的教学 CPU 采用自己的简化指令格式，具体位分配以课程电路为准。

## 3. 汇编语言：机器指令的可读写法

直接书写二进制机器码很容易出错，因此人们用助记符表示指令：

```asm
LOAD_A 10
LOAD_B 11
ADD
STORE 12
```

这里的 `LOAD_A`、`LOAD_B`、`ADD`、`STORE` 是汇编助记符，后面的数字可能表示内存地址或立即数。汇编器会把这些文本转换成 CPU 能识别的机器码。

### 3.1 汇编与机器码的关系

| 层次 | 示例 | 处理者 |
|---|---|---|
| 汇编语言 | `LOAD_A 10` | 程序员和汇编器 |
| 机器码 | 一串 0/1 或十六进制字节 | CPU |
| 电路动作 | 读内存、装载寄存器、改变控制线 | CPU 内部硬件 |

助记符不是新的硬件能力，它只是机器指令编码的可读名称。

## 4. 一条指令需要哪些硬件状态

为了执行一条指令，CPU 至少需要保存几个阶段的状态：

- **程序计数器（PC）：**指出下一条指令所在的内存地址；
- **指令寄存器（IR）：**保存当前正在解释的指令；
- **操作数寄存器：**暂存从内存取出的数据；
- **ALU 输入/结果寄存器：**为运算提供操作数并保存中间结果；
- **标志或状态寄存器：**保存进位、溢出、零等状态（是否存在取决于教学 CPU）。

取指和执行可以抽象为：

```text
PC 给出地址 → 内存输出指令 → IR 保存指令
        → 控制逻辑解释操作码 → 读取操作数 → ALU/寄存器完成动作
```

本篇先用“手动控制”观察这个过程；下一篇再让控制器自动产生时序信号。

## 5. 内存模块的使用

内存为 CPU 提供两个基本操作：

### 读取

```text
地址总线 → 选择某个单元
读控制有效 → 数据总线输出该单元的内容
```

### 写入

```text
地址总线 → 选择某个单元
数据总线 → 放入待写数据
写控制有效 → 在规定时刻保存
```

### 5.1 内存访问的三个要素

| 要素 | 作用 |
|---|---|
| 地址 | 指定访问哪个单元 |
| 数据 | 要读出的内容或要写入的内容 |
| 控制 | 区分读、写、片选和时序 |

CPU 不能只把数据线接上就使用内存，还必须保证地址和控制信号在正确时间稳定。

## 6. CPU 与内存的连接

CPU 和内存通常通过三类总线连接：

| 总线 | 方向 | 作用 |
|---|---|---|
| 地址总线 | CPU → 内存 | 指定内存单元 |
| 数据总线 | 双向 | 传输指令和数据 |
| 控制总线 | 多方向 | 读、写、使能、时钟等控制 |

抽象连接如下：

```text
             地址总线 ───────────> 内存地址输入
CPU 内部 ───< 数据总线 ─────────> 内存数据端口
             控制总线 ───────────> 读/写/片选
```

数据总线是双向的：取指或读数据时由内存驱动，写数据时由 CPU 侧驱动。两种方向不能同时有效，否则会发生总线争用。

## 7. 手动版本 CPU 的组成

课程视频 64 完成手动版本 CPU 的搭建。这里的“手动”不是没有 CPU，而是由人使用开关或控制面板代替自动控制器，逐步发出控制信号。

手动版本通常包括：

- 程序计数器或地址输入；
- 指令/数据寄存器；
- A、B 等工作寄存器；
- ALU；
- 内部数据总线；
- 内存模块；
- 输出指示和手动控制开关。

手动操作的价值是让每一次数据传输都可见：按下一个控制开关，观察某个寄存器输出；切换另一个信号，观察目标寄存器加载。自动执行虽然方便，但会隐藏这些底层动作。

## 8. 手动执行第一条指令：`LOAD_A`

`LOAD_A` 的含义是把内存中指定位置的数据读入 A 寄存器。可以把它拆成以下动作：

1. 把指令所在地址送到地址总线；
2. 读内存，取出 `LOAD_A` 指令；
3. 把指令装入指令寄存器；
4. 解析指令中的操作数地址；
5. 把操作数地址送到内存地址端；
6. 再次读内存，得到待加载的数据；
7. 让内存数据驱动内部总线；
8. 在时钟边沿让 A 寄存器加载总线数据。

抽象数据流：

```text
指令地址 → 内存 → 指令寄存器
操作数地址 → 内存 → 总线 → A 寄存器
```

教学电路中具体按钮顺序可能按面板设计略有不同，但“地址选择—内存读出—总线传送—寄存器加载”的逻辑不变。

## 9. 手动执行 `LOAD_A` 的时序表

可以用时序表记录一次手动执行，避免漏掉某个控制信号：

| 步骤 | 地址/数据动作 | 关键控制 | 结果 |
|---|---|---|---|
| 1 | 选择指令地址 | 地址输出、内存读 | 内存输出指令 |
| 2 | 保存指令 | 指令寄存器 LOAD | IR 保存 `LOAD_A` |
| 3 | 选择操作数地址 | 地址字段送地址总线 | 选中数据单元 |
| 4 | 读取数据 | 内存读、内存输出 | 数据出现在总线 |
| 5 | 保存到 A | A 寄存器 LOAD、时钟边沿 | A 得到操作数 |

排错时可以逐行观察：地址是否正确、内存输出是否稳定、总线是否被正确驱动、A 的 `LOAD` 是否在正确边沿有效。

## 10. 手动完成 `3+5` 的整体流程

课程视频 66 用一个简单加法展示 CPU 如何执行程序。这里用抽象汇编表示：

```asm
LOAD_A 3       ; 把常数 3 装入 A
LOAD_B 5       ; 把常数 5 装入 B
ADD             ; A+B，结果进入结果寄存器/累加器
```

不同教学 CPU 可能把操作数放在内存单元，也可能把它们编码为立即数；重点是执行流程：

1. 取出第一条 `LOAD_A`，读出 3，装入 A；
2. 取出第二条 `LOAD_B`，读出 5，装入 B；
3. 取出 `ADD`，让 A、B 进入 ALU；
4. ALU 计算 `3+5=8`；
5. 让结果通过总线写回目标寄存器；
6. 若需要，再把结果写入内存或输出设备。

### 10.1 为什么要多条指令

ALU 只负责当前被要求的运算，不能自动知道数据来自哪里。`LOAD` 指令负责准备操作数，`ADD` 指令负责选择运算，寄存器和总线负责保存、传递结果。

## 11. `3+5` 的寄存器和总线视角

从数据通路看，`3+5` 可以拆成几次寄存器传送：

```text
内存[常数3] → 总线 → A
内存[常数5] → 总线 → B
A、B → ALU(ADD) → 总线 → 结果寄存器
```

每一次箭头都需要对应的输出使能、加载控制和时钟边沿。

### 11.1 手动执行的观察重点

- 执行 `LOAD_A` 后，A 是否为 3；
- 执行 `LOAD_B` 后，B 是否为 5；
- `ADD` 期间，ALU 的运算选择是否为加法；
- 结果写回时，是否只有 ALU 结果驱动总线；
- 最终结果是否为 8，进位和溢出是否符合位宽规则。

这个例子把第 05 篇的数据通路与本篇的机器指令连接起来：指令只是控制数据通路按特定顺序动作。

## 12. 程序计数器和顺序取指

连续执行多条指令需要知道“下一条在哪里”。程序计数器（PC）保存当前或下一条指令的地址。每完成一次取指，PC 通常增加到下一条地址；遇到跳转指令时，PC 可能被改写为另一个地址。

本篇的视频重点是手动执行，PC 的每一步可以由人观察或设置。自动控制器加入后，会在时钟周期中自动完成：

```text
PC 输出地址 → 内存读指令 → 指令寄存器加载 → PC 更新
```

即使课程中的教学 CPU 对 PC 的具体实现有所简化，理解“指令地址必须按顺序变化”仍是连接内存和指令执行的关键。

## 13. 从 C 代码到 CPU 执行

课程视频 67 解释 C 代码执行时的层次变化。以：

```c
int result = 3 + 5;
```

为例，可以经过以下层次：

```text
C 源代码
  ↓ 编译器
汇编代码
  ↓ 汇编器
机器指令/目标文件
  ↓ 装入内存
CPU 取指、译码、执行
  ↓
寄存器、ALU、内存和输出设备发生硬件动作
```

编译器可能优化表达式，实际生成的指令不一定与示例完全相同。学习底层原理时，重点是理解抽象层之间的关系：高级语言表达意图，编译器选择指令，CPU 只执行指令集规定的动作。

## 14. 变量、内存和寄存器的关系

C 语言中的变量是程序员看到的概念；在执行过程中，它可能暂存在寄存器，也可能位于内存。

- **寄存器：**访问快，数量少，适合当前正在计算的值；
- **内存：**容量大，适合保存程序和更多变量；
- **地址：**让 CPU 能够找到变量所在的存储单元；
- **加载/存储：**在内存和寄存器之间搬运数据。

因此，`result = a + b` 底层通常需要“从某处取出 a、b → ALU 相加 → 把结果放回某处”这一类动作。具体指令数量和使用哪些寄存器由编译器、指令集和优化策略决定。

## 15. 显存的工作原理

显存可以看成一块被显示控制器解释的特殊内存。CPU 或其他处理单元把像素数据写入显存，显示控制器按规定的行列顺序读取这些数据，并转换为屏幕上的颜色和亮度。

抽象流程：

```text
CPU/程序 → 写入显存地址 → 显示控制器读取
                              ↓
                         像素/字符输出到屏幕
```

### 15.1 显存中的数据

不同系统可能用不同格式表示像素：

- 1 位表示黑白像素；
- 多位表示灰度；
- 多个字节表示 RGB 或带透明度的颜色；
- 文本模式下，内存单元可以保存字符编码和属性。

本课程的重点是理解“写某个内存位置会影响显示内容”，而不是绑定到某一种现代显卡协议。

## 16. 显存和普通内存的区别

显存与普通内存在电路层面都可以是可寻址存储，但它们的使用方式不同：

| 对比点 | 普通内存 | 显存/显示缓冲区 |
|---|---|---|
| 内容 | 指令、变量、栈等 | 像素或字符显示数据 |
| 读取者 | CPU、其他硬件模块 | 显示控制器持续读取 |
| 写入效果 | 改变程序状态 | 可能改变屏幕内容 |
| 组织方式 | 按地址保存通用数据 | 按像素、行列或字符布局解释 |

显示控制器持续扫描显存并刷新屏幕，所以 CPU 写入显存后，显示结果通常会在下一次刷新中体现。若 CPU 和显示控制器同时访问同一块存储，还需要专门的仲裁或带宽设计。

## 17. 指令执行的统一观察框架

无论是 `LOAD_A`、`ADD` 还是写显存，都可以按四个问题分析：

1. **取什么：**当前指令和操作数位于哪个地址？
2. **经过哪里：**数据通过哪个总线、寄存器或 ALU？
3. **谁控制：**哪些 OE、LOAD、读写和运算选择信号有效？
4. **何时保存：**在哪个时钟边沿把结果写入寄存器或内存？

用这个框架阅读电路图、手动操作面板或分析汇编代码，可以把“执行一条指令”从抽象描述还原成具体硬件步骤。

## 18. 常见问题与排错方法

### 指令和地址

- 把操作码和操作数地址混淆：先明确指令格式；
- PC 指向错误地址：检查取指后是否正确更新；
- 把数据当成指令：确认程序区和数据区的地址安排。

### 内存和总线

- 地址对了但读出的数据不对：检查内存读控制、片选和数据总线方向；
- 写入后数据消失：检查写入时刻、写使能和目标单元；
- 总线出现冲突：确保同一时刻只有一个输出端有效。

### 运算和显示

- `3+5` 结果错误：分别观察 A、B、ALU 运算选择和结果寄存器；
- 显存写入无显示：检查显存地址、像素格式和显示控制器读取范围；
- C 代码和示例指令不同：先确认编译器可能进行了优化，不要把某个汇编序列当成唯一结果。

## 19. 一页复习表

| 知识点 | 关键结论 |
|---|---|
| 指令集 | CPU 支持的操作码、操作数格式和行为集合 |
| 机器码 | 指令在硬件中使用的二进制编码 |
| 汇编 | 机器指令的可读助记符 |
| PC | 保存下一条或当前指令的地址 |
| 指令寄存器 | 保存正在执行的指令 |
| 地址总线 | 选择内存单元 |
| 数据总线 | 在 CPU 与内存之间传输数据/指令 |
| 控制总线 | 读、写、使能和时钟等控制 |
| `LOAD_A` | 从内存取操作数并装入 A 寄存器 |
| `3+5` | 取指 → 加载操作数 → ALU 加法 → 结果写回 |
| C 代码执行 | 源代码 → 汇编 → 机器码 → 硬件动作 |
| 显存 | 被显示控制器解释的显示数据存储区 |

## 20. 自测题与参考答案

1. 指令集、机器码和汇编语言分别是什么？
2. `LOAD_A` 指令至少需要完成哪些硬件动作？
3. CPU 与内存之间为什么需要地址总线、数据总线和控制总线？
4. 手动执行 `3+5` 时，为什么要先执行加载指令再执行 `ADD`？
5. 程序计数器的作用是什么？
6. C 代码是怎样逐层变成 CPU 可以执行的动作的？
7. 显存为什么可以影响屏幕显示？
8. 如果 `LOAD_A` 读到了错误数据，应从哪些方面排查？

### 参考答案

1. 指令集是 CPU 支持的指令规则集合；机器码是二进制编码；汇编是机器码的助记符表示。
2. 取指、解析操作数地址、读内存、让数据进入总线，并在时钟边沿加载 A 寄存器。
3. 地址总线选择位置，数据总线传输内容，控制总线规定读写和时序。
4. ALU 只处理已经送入输入通路的操作数，必须先把 3 和 5 放入相应寄存器或输入端。
5. 它保存下一条或当前指令的地址，支持顺序取指和后续控制流。
6. 源代码经编译生成汇编/机器码，装入内存后由 CPU 取指、译码、执行。
7. 显示控制器把显存中的数据按像素或字符规则读取并转换为屏幕信号。
8. 检查指令地址、操作数地址、内存读控制、数据总线方向、A 的 LOAD 和时钟边沿。

## 21. 与原课程资料对应关系

| 本篇内容 | 视频 | 主题 |
|---|---|---|
| 汇编语言和指令集 | 61 | 指令、机器码、助记符和指令集 |
| 内存模块的使用 | 62 | 地址、数据、读写控制 |
| CPU 与内存连接 | 63 | 地址总线、数据总线、控制总线 |
| 手动版本 CPU 完成 | 64 | 用面板控制 CPU 内部动作 |
| 手动执行 `LOAD_A` | 65 | 取指、读操作数、装载寄存器 |
| 手动完成 `3+5` | 66 | 加载操作数、ALU 运算、结果写回 |
| C 代码执行解释 | 67 | 源代码到机器动作的抽象层次 |
| 显存工作原理 | 68 | 显存、显示控制器和屏幕输出 |

**资料来源：**《自己动手做台计算机》课程，百度网盘分享链接：`https://pan.baidu.com/s/1RLiC8gC1sXTu7EgkQs9oHw?pwd=6633`，提取码：`6633`。本篇按视频 61—68 整理；第 69 个视频起的自动控制单元放入第 07 篇。